In [ ]:
from google.colab import drive

drive.mount("/content/drive")

In [ ]:
# Create directory
!mkdir -p /content/bobiac_data_cellpose
# Download the data
!wget https://raw.githubusercontent.com/bobiac/bobiac-book/main/_static/data/05_segmentation_cellpose_training.zip -O /content/bobiac_data_cellpose/05_segmentation_cellpose_training.zip
# Unzip the data, remove zip file and macOS metadata files (if any)
!cd /content/bobiac_data_cellpose && unzip 05_segmentation_cellpose_training.zip && rm -f 05_segmentation_cellpose_training.zip && rm -rf __MACOSX

In [ ]:
# !pip install cellpose

In [3]:
from pathlib import Path

import numpy as np
from cellpose import core, io, metrics, models, train

In [ ]:
io.logger_setup()  # to get printing of progress

use_gpu = core.use_gpu()
print("GPU available:", use_gpu)

In [ ]:
ROOT_FOLDER_PATH = Path("data/05_segmentation_cellpose/retraining")

train_dir = ROOT_FOLDER_PATH / "train"
test_dir = ROOT_FOLDER_PATH / "test"

masks_ext = "_seg.npy"

# get files
train_data, train_labels, _, test_data, test_labels, _ = io.load_train_test_data(
    train_dir, test_dir, mask_filter=masks_ext
)

In [ ]:
# Convert images to float32
train_data = []
for img in train_data:
    train_data.append(img.astype(np.float32))
# same as using list comprehension:
# train_data = [img.astype(np.float32) for img in train_data]

# Convert labels (masks) to int32
train_labels = []
for lbl in train_labels:
    train_labels.append(lbl.astype(np.int32))
# same as using list comprehension:
# train_labels = [lbl.astype(np.int32) for lbl in train_labels]

# Convert test images to float32 and labels to int32
test_data = []
for img in test_data:
    test_data.append(img.astype(np.float32))
# same as using list comprehension:
# test_data = [img.astype(np.float32) for img in test_data]

# Convert test labels (masks) to int32
test_labels = []
for lbl in test_labels:
    test_labels.append(lbl.astype(np.int32))
# same as using list comprehension:
# test_labels = [lbl.astype(np.int32) for lbl in test_labels]

In [ ]:
# Initialize the Cellpose model
model = models.CellposeModel(gpu=use_gpu, model_type="cpsam")

In [ ]:
# run model on test images
masks = model.eval(test_data, batch_size=32)[0]

# check performance using ground truth labels
ap = metrics.average_precision(test_labels, masks)[0]
print(f"\n>>> average precision at iou threshold 0.5 = {ap[:, 0].mean():.3f}")

In [ ]:
model_name = "new_model"

# Training params
n_epochs = 10
learning_rate = 1e-5
weight_decay = 0.1
batch_size = 1

# (not passing test data into function to speed up training)

new_model_path, train_losses, test_losses = train.train_seg(
    model.net,
    train_data=train_data,
    train_labels=train_labels,
    batch_size=batch_size,
    n_epochs=n_epochs,
    learning_rate=learning_rate,
    weight_decay=weight_decay,
    nimg_per_epoch=max(2, len(train_data)),  # can change this
    model_name=model_name,
)

In [ ]:
model = models.CellposeModel(gpu=True, pretrained_model=new_model_path)

# run model on test images
masks = model.eval(test_data, batch_size=32)[0]

# check performance using ground truth labels
ap = metrics.average_precision(test_labels, masks)[0]
print(f"\n>>> average precision at iou threshold 0.5 = {ap[:, 0].mean():.3f}")